**Programmer: python_scripts (Abhijith Warrier)**

**🔬 Python Code Lab — *Generator vs List Measuring Real Memory Usage***

Compare fully consumed workloads, not just empty generator objects. We measure **peak Python allocations tracked by tracemalloc**, not total process RAM (RSS). All three paths must produce the same sum.

---
### 1. Define the workloads and measurement
Start a fresh trace for each run, before allocation. Read peak memory after consumption: the peak retains the temporary list's high-water mark even after the list has been released. Collection, printing and assertions happen outside the measured region.

In [1]:
import gc
import sys
import json
import platform
import statistics
import tracemalloc
from pathlib import Path

# Both paths create and consume exactly the same sequence.
def list_total(n):
    return sum([i * i for i in range(n)])

def generator_total(n):
    return sum(i * i for i in range(n))

def materialized_total(n):
    return sum(list(i * i for i in range(n)))

def measure_peak(fn, n):
    if tracemalloc.is_tracing():
        raise RuntimeError("Run in a kernel without an existing tracing session")
    gc.collect()
    tracemalloc.start()
    try:
        result = fn(n)
        _, peak = tracemalloc.get_traced_memory()
    finally:
        tracemalloc.stop()
    expected = n * (n - 1) * (2 * n - 1) // 6
    assert result == expected
    return peak


---
### 2. Increase the input size
Run each case three times and report the median peak. `range` avoids prebuilding an input list. All repeated measurements are saved; exact sizes depend on Python and the environment. Tracing is process-wide: background notebook activity may add noise, so inspect the saved raw repeats as well as the median.

In [2]:
sizes = [10_000, 100_000, 1_000_000]
functions = {"List": list_total, "Generator": generator_total,
             "Materialized generator": materialized_total}
rows = []
for n in sizes:
    for name, fn in functions.items():
        peaks = [measure_peak(fn, n) for _ in range(3)]
        rows.append({"n": n, "method": name,
                     "peak_bytes": int(statistics.median(peaks)),
                     "repeats_bytes": peaks})
        print(f"{n:>9,} | {name:>22} | {statistics.median(peaks):>12,.0f} bytes")
report = {"python": platform.python_version(), "implementation": platform.python_implementation(),
          "platform": platform.platform(), "metric": "median of 3 peak traced allocations in bytes", "rows": rows}
Path("memory_results.json").write_text(json.dumps(report, indent=2))


   10,000 |                   List |      404,864 bytes
   10,000 |              Generator |          624 bytes
   10,000 | Materialized generator |      405,072 bytes


  100,000 |                   List |    4,000,672 bytes
  100,000 |              Generator |          472 bytes


  100,000 | Materialized generator |    4,001,087 bytes


1,000,000 |                   List |   40,448,416 bytes


1,000,000 |              Generator |          472 bytes


1,000,000 | Materialized generator |   40,448,624 bytes


1711

---
### 3. Why shallow size is not enough
`sys.getsizeof` measures the object itself, not all objects it references. A generator also holds execution state and may retain external objects. Our generator is consumed once; it does not store every square.

In [3]:
values = [i * i for i in range(10_000)]
stream = (i * i for i in range(10_000))
print("List shallow size:", sys.getsizeof(values), "bytes")
print("Generator shallow size:", sys.getsizeof(stream), "bytes")
# getsizeof does not recursively count referenced objects.
stream.close()
del values, stream

stream = (i * i for i in range(5))
assert sum(stream) == 30
assert sum(stream) == 0  # The iterator is exhausted.


List shallow size: 85176 bytes
Generator shallow size: 208 bytes


---
### 4. Visualize the peak
Both axes use logarithmic scales to show the small generator trace alongside the lists. The two list-based curves may overlap.

Lists retain every generated integer; this streaming sum retains only limited working state. Wrapping the generator in `list(...)` materializes everything and loses that advantage. Generators are single-pass; choose lists when indexing or reuse is needed. This is a memory experiment, not a speed benchmark.

`tracemalloc` does not include the whole interpreter, its own tracking overhead, or every native allocation. Streaming memory is not universally constant: retained inputs, buffering, and growing objects matter.

[tracemalloc](https://docs.python.org/3.11/library/tracemalloc.html) · [getsizeof](https://docs.python.org/3.11/library/sys.html#sys.getsizeof) · [Generators](https://docs.python.org/3.11/tutorial/classes.html#generators)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4.5))
for name in functions:
    selected = [r for r in rows if r["method"] == name]
    ax.plot([r["n"] for r in selected],
            [r["peak_bytes"] / 1024**2 for r in selected], marker="o", label=name)
ax.set(xscale="log", yscale="log", xlabel="Number of squared integers (log scale)",
       ylabel="Peak traced allocations, MiB (log scale)", title="Create + fully consume: same sum")
ax.legend(); ax.grid(alpha=.2); fig.tight_layout()
fig.savefig("memory_comparison.png", dpi=180)
plt.show()


---